In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import xml.etree.ElementTree as ET
import json
import os
import re

In [ ]:
# Main folder containing all frame folders
BASE_FOLDER = "/content/drive/MyDrive/Frames_experiment" #change to your folder
OUTPUT_FILE = "/content/drive/MyDrive/Frames_experiment/doccano_input.jsonl"

In [ ]:
def clean_text(text):
    text = text.replace("<s>", "").replace("</s>", "")
    text = re.sub(r"/[a-zA-Z0-9\.\-]+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
sentences = []

for frame_folder in os.listdir(BASE_FOLDER):

    frame_path = os.path.join(BASE_FOLDER, frame_folder)

    if os.path.isdir(frame_path):

        for file in os.listdir(frame_path):

            if file.endswith(".xml"):

                file_path = os.path.join(frame_path, file)

                tree = ET.parse(file_path)
                root = tree.getroot()

                for line in root.findall(".//line"):

                    left = line.find("left")
                    kwic = line.find("kwic")
                    right = line.find("right")

                    if left is None or kwic is None or right is None:
                        continue

                    left_text = left.text if left.text else ""
                    kwic_text = kwic.text if kwic.text else ""
                    right_text = right.text if right.text else ""

                    full_text = left_text + " " + kwic_text + " " + right_text
                    full_text = clean_text(full_text)

                    if full_text:
                        sentences.append(full_text)

print("Total sentences extracted:", len(sentences))

In [ ]:
with open(OUTPUT_FILE, "w", encoding="utf-8") as out_file:

    for sentence in sentences:
        json.dump({"text": sentence}, out_file, ensure_ascii=False)
        out_file.write("\n")

print("Doccano input file created:")
print(OUTPUT_FILE)